In [1]:
import torch
from torch import nn

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [5]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [7]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
logits

tensor([[-0.0368,  0.0114, -0.1004, -0.1078,  0.0017, -0.0605,  0.0451,  0.0280,
         -0.0295,  0.0265]], grad_fn=<AddmmBackward0>)

In [8]:
pred_proba = nn.Softmax(dim=1)(logits)
y_pred = pred_proba.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([6])


In [9]:
input_image = torch.rand(3,28,28)
print(input_image.size())
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 28, 28])
torch.Size([3, 784])


In [10]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [11]:
print(f"Before ReLU: {hidden1}\n")
hidden1 = nn.ReLU()(hidden1)
print(f"after ReLU: {hidden1}")

Before ReLU: tensor([[-0.1166, -0.5035, -0.0912,  0.2129,  0.2771,  0.2102,  0.1248, -0.5962,
          0.2690,  0.5487, -0.2286, -0.3051, -0.1039, -0.2365, -0.3595, -0.4071,
         -0.2590, -0.1058,  0.1152, -0.4647],
        [-0.2616, -1.0834,  0.3105,  0.1903,  0.3930,  0.1734,  0.1781, -0.4976,
          0.1151,  0.4403, -0.1752, -0.0893, -0.0013, -0.2801,  0.0581, -0.3601,
         -0.7858,  0.3047,  0.0468, -0.3682],
        [-0.0777, -0.5331,  0.1391, -0.0128,  0.4190,  0.2638, -0.1156, -0.2851,
          0.2742,  0.4424, -0.1130, -0.3718, -0.1384, -0.1296, -0.2308, -0.6507,
         -0.5113,  0.3848,  0.0666, -0.4431]], grad_fn=<AddmmBackward0>)

after ReLU: tensor([[0.0000, 0.0000, 0.0000, 0.2129, 0.2771, 0.2102, 0.1248, 0.0000, 0.2690,
         0.5487, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.1152, 0.0000],
        [0.0000, 0.0000, 0.3105, 0.1903, 0.3930, 0.1734, 0.1781, 0.0000, 0.1151,
         0.4403, 0.0000, 0.0000, 0.0000, 0.0000, 0.058

In [12]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)
softmax = nn.Softmax(dim=1)
pred_proba = softmax(logits)
pred_proba

tensor([[0.0921, 0.1030, 0.0775, 0.0875, 0.0939, 0.1053, 0.0927, 0.1190, 0.1268,
         0.1022],
        [0.0906, 0.1051, 0.0764, 0.0808, 0.1061, 0.0979, 0.0911, 0.1168, 0.1178,
         0.1174],
        [0.0957, 0.1016, 0.0792, 0.0911, 0.0935, 0.0937, 0.0977, 0.1151, 0.1263,
         0.1060]], grad_fn=<SoftmaxBackward0>)

In [13]:
print(f"Model structure: {model}\n\n")
for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0200,  0.0076, -0.0190,  ...,  0.0031, -0.0292,  0.0051],
        [ 0.0254,  0.0300, -0.0110,  ...,  0.0263,  0.0049,  0.0048]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0048, -0.0185], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0230,  0.0213, -0.0419,  ...,  0.0423, -0.0062,  0.0272],
        [-0.0181,  0.0093, -0.0028,  ..., -0.0099, -0.0193,  0.0080]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | 